In [18]:
# ============================================================
# IMPORTS
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.4f}'.format)



In [19]:
# ============================================================
# LOAD DATA
# ============================================================
# Load
df = pd.read_csv("clean_events.csv")
print(df)


         visitorid session_id  session_num event_type  itemid  transactionid  \
0                6        6_1            1  addtocart   65273            NaN   
1                6        6_2            2       view  253615            NaN   
2                6        6_2            2       view  344723            NaN   
3                6        6_2            2       view  344723            NaN   
4                6        6_2            2       view  344723            NaN   
...            ...        ...          ...        ...     ...            ...   
2755636    1407561  1407561_1            1       view   89871            NaN   
2755637    1407566  1407566_1            1       view  357369            NaN   
2755638    1407567  1407567_1            1       view  219086            NaN   
2755639    1407571  1407571_1            1       view  113338            NaN   
2755640    1407573  1407573_2            2       view  232069            NaN   

                     event_datetime  ho

In [20]:
# ============================================================
# DATA PREVIEW
# ============================================================

# FIRST 5 ROWS
display(df.head())

# DATA TYPES
print(df.dtypes)

print("NULL CHECK")
print(df.isnull().sum())

# EVENT TYPE DISTRIBUTION
print(df['event_type'].value_counts())

,visitorid,session_id,session_num,event_type,itemid,transactionid,event_datetime,hour_of_day,day_of_week,is_weekend,categoryid,available
0,6,6_1,1,addtocart,65273,NaN,2015-08-30 12:03:48.202+06,12,0,1,342,1
1,6,6_2,2,view,253615,NaN,2015-08-30 12:39:38.318+06,12,0,1,342,1
2,6,6_2,2,view,344723,NaN,2015-08-30 12:40:23.805+06,12,0,1,342,1
3,6,6_2,2,view,344723,NaN,2015-08-30 12:54:09.385+06,12,0,1,342,1
4,6,6_2,2,view,344723,NaN,2015-08-30 12:56:33.838+06,12,0,1,342,1


visitorid           int64
session_id            str
session_num         int64
event_type            str
itemid              int64
transactionid     float64
event_datetime        str
hour_of_day         int64
day_of_week         int64
is_weekend          int64
categoryid          int64
available           int64
dtype: object
NULL CHECK
visitorid               0
session_id              0
session_num             0
event_type              0
itemid                  0
transactionid     2733184
event_datetime          0
hour_of_day             0
day_of_week             0
is_weekend              0
categoryid              0
available               0
dtype: int64
event_type
view           2664218
addtocart        68966
transaction      22457
Name: count, dtype: int64


In [21]:
# Convert datetime (mixed format fix)
df['event_datetime'] = pd.to_datetime(df['event_datetime'], format='mixed')

# Remove timezone info
df['event_datetime'] = df['event_datetime'].dt.tz_localize(None)

# Convert IDs to string
df['visitorid']  = df['visitorid'].astype(str)
df['session_id'] = df['session_id'].astype(str)
df['itemid']     = df['itemid'].astype(str)

# Convert numeric columns
df['hour_of_day'] = df['hour_of_day'].astype(int)
df['day_of_week'] = df['day_of_week'].astype(int)
df['is_weekend']  = df['is_weekend'].astype(int)

# Verify
print(df.dtypes)
print(df['event_datetime'].head())

visitorid                    str
session_id                   str
session_num                int64
event_type                   str
itemid                       str
transactionid            float64
event_datetime    datetime64[us]
hour_of_day                int64
day_of_week                int64
is_weekend                 int64
categoryid                 int64
available                  int64
dtype: object
0   2015-08-30 12:03:48.202
1   2015-08-30 12:39:38.318
2   2015-08-30 12:40:23.805
3   2015-08-30 12:54:09.385
4   2015-08-30 12:56:33.838
Name: event_datetime, dtype: datetime64[us]


In [22]:
# ============================================================
# SECTION 1: SESSION-LEVEL FEATURES
# ============================================================

# Create binary flags (IMPORTANT for speed)
df['is_view'] = (df['event_type'] == 'view').astype(int)
df['is_cart'] = (df['event_type'] == 'addtocart').astype(int)
df['is_purchase'] = (df['event_type'] == 'transaction').astype(int)

# Session aggregation (FAST)
session_features = df.groupby('session_id').agg(
    visitorid=('visitorid', 'first'),
    events_per_session=('event_type', 'count'),
    unique_items_per_session=('itemid', 'nunique'),
    unique_event_types=('event_type', 'nunique'),

    has_view=('is_view', 'max'),
    has_cart=('is_cart', 'max'),
    has_purchase=('is_purchase', 'max'),

    session_start=('event_datetime', 'min'),
    session_end=('event_datetime', 'max'),

    session_hour_start=('hour_of_day', 'first'),
    session_day_of_week=('day_of_week', 'first'),
    is_weekend_session=('is_weekend', 'first')
).reset_index()

# Session duration
session_features['session_duration_mins'] = (
    session_features['session_end'] - session_features['session_start']
).dt.total_seconds() / 60

# Diversity score
session_features['event_diversity_score'] = (
    session_features['unique_event_types'] / 3
).round(4)

# Outcome label (FAST)
import numpy as np

session_features['session_outcome'] = np.select(
    [
        session_features['has_purchase'] == 1,
        session_features['has_cart'] == 1,
        session_features['events_per_session'] <= 2
    ],
    [
        'purchase',
        'cart_abandon',
        'bounce'
    ],
    default='browse'
)

In [23]:
# ============================================================
# SECTION 2: USER-LEVEL FEATURES (OPTIMIZED)
# ============================================================

import numpy as np

# ----------------------------
# Step 1: Binary flags (FAST)
# ----------------------------
df['is_view'] = (df['event_type'] == 'view').astype(int)
df['is_cart'] = (df['event_type'] == 'addtocart').astype(int)
df['is_purchase'] = (df['event_type'] == 'transaction').astype(int)

# ----------------------------
# Dataset end date
# ----------------------------
dataset_end = df['event_datetime'].max()

# ----------------------------
# Step 2: User aggregation
# ----------------------------
user_features = df.groupby('visitorid').agg(

    total_sessions=('session_id', 'nunique'),

    total_events=('event_type', 'count'),
    total_views=('is_view', 'sum'),
    total_carts=('is_cart', 'sum'),
    total_purchases=('is_purchase', 'sum'),

    unique_items_viewed=('itemid', 'nunique'),
    unique_categories_viewed=('categoryid', 'nunique'),

    first_visit=('event_datetime', 'min'),
    last_visit=('event_datetime', 'max'),

    preferred_hour=('hour_of_day', 'median'),
    weekend_session_pct=('is_weekend', 'mean')

).reset_index()

# ----------------------------
# Step 3: Recency features
# ----------------------------
user_features['days_since_first_visit'] = (dataset_end - user_features['first_visit']).dt.days
user_features['days_since_last_visit'] = (dataset_end - user_features['last_visit']).dt.days

# ----------------------------
# Step 4: Ratio features
# ----------------------------
user_features['view_to_cart_ratio'] = (
    user_features['total_carts'] /
    user_features['total_views'].replace(0, np.nan)
).round(4)

user_features['cart_to_purchase_ratio'] = (
    user_features['total_purchases'] /
    user_features['total_carts'].replace(0, np.nan)
).round(4)

# ----------------------------
# Step 5: Binary flags
# ----------------------------
user_features['is_weekend_user'] = (user_features['weekend_session_pct'] >= 0.5).astype(int)
user_features['is_repeat_buyer'] = (user_features['total_purchases'] > 1).astype(int)
user_features['has_purchased'] = (user_features['total_purchases'] > 0).astype(int)
user_features['has_cart'] = (user_features['total_carts'] > 0).astype(int)

# ----------------------------
# Step 6: Purchase probability score
# ----------------------------
user_features['purchase_probability_score'] = (
    user_features['view_to_cart_ratio'].fillna(0) *
    user_features['cart_to_purchase_ratio'].fillna(0)
).round(4)

# ----------------------------
# Verify
# ----------------------------
print(user_features.shape)
user_features.head()

(1407580, 21)


,visitorid,total_sessions,total_events,total_views,total_carts,total_purchases,unique_items_viewed,unique_categories_viewed,first_visit,last_visit,preferred_hour,weekend_session_pct,days_since_first_visit,days_since_last_visit,view_to_cart_ratio,cart_to_purchase_ratio,is_weekend_user,is_repeat_buyer,has_purchased,has_cart,purchase_probability_score
0,0,1,3,3,0,0,3,3,2015-09-12 02:49:49.439,2015-09-12 02:55:17.175,2.0000,1.0000,6,6,0.0000,NaN,1,0,0,0,0.0000
1,1,1,1,1,0,0,1,1,2015-08-13 23:46:06.444,2015-08-13 23:46:06.444,23.0000,0.0000,35,35,0.0000,NaN,0,0,0,0,0.0000
2,10,1,1,1,0,0,1,1,2015-08-05 00:30:29.611,2015-08-05 00:30:29.611,0.0000,0.0000,44,44,0.0000,NaN,0,0,0,0,0.0000
3,100,1,4,4,0,0,1,1,2015-09-09 05:48:55.990,2015-09-09 05:52:54.884,5.0000,0.0000,9,9,0.0000,NaN,0,0,0,0,0.0000
4,1000,1,1,1,0,0,1,1,2015-07-05 19:38:36.161,2015-07-05 19:38:36.161,19.0000,1.0000,74,74,0.0000,NaN,1,0,0,0,0.0000


In [24]:
# ============================================================
# SECTION 2: USER-LEVEL FEATURES (OPTIMIZED)
# ============================================================

# ----------------------------
# Session avg per user (FAST)
# ----------------------------
session_avg = session_features.groupby('visitorid', as_index=False).agg(
    avg_session_duration=('session_duration_mins', 'mean'),
    avg_events_per_session=('events_per_session', 'mean')
)

# ----------------------------
# Merge into user features
# ----------------------------
user_features = user_features.merge(
    session_avg,
    on='visitorid',
    how='left'
)

# ----------------------------
# Verify
# ----------------------------
print(user_features.shape)
user_features.head()

(1407580, 23)


,visitorid,total_sessions,total_events,total_views,total_carts,total_purchases,unique_items_viewed,unique_categories_viewed,first_visit,last_visit,preferred_hour,weekend_session_pct,days_since_first_visit,days_since_last_visit,view_to_cart_ratio,cart_to_purchase_ratio,is_weekend_user,is_repeat_buyer,has_purchased,has_cart,purchase_probability_score,avg_session_duration,avg_events_per_session
0,0,1,3,3,0,0,3,3,2015-09-12 02:49:49.439,2015-09-12 02:55:17.175,2.0000,1.0000,6,6,0.0000,NaN,1,0,0,0,0.0000,5.4623,3.0000
1,1,1,1,1,0,0,1,1,2015-08-13 23:46:06.444,2015-08-13 23:46:06.444,23.0000,0.0000,35,35,0.0000,NaN,0,0,0,0,0.0000,0.0000,1.0000
2,10,1,1,1,0,0,1,1,2015-08-05 00:30:29.611,2015-08-05 00:30:29.611,0.0000,0.0000,44,44,0.0000,NaN,0,0,0,0,0.0000,0.0000,1.0000
3,100,1,4,4,0,0,1,1,2015-09-09 05:48:55.990,2015-09-09 05:52:54.884,5.0000,0.0000,9,9,0.0000,NaN,0,0,0,0,0.0000,3.9816,4.0000
4,1000,1,1,1,0,0,1,1,2015-07-05 19:38:36.161,2015-07-05 19:38:36.161,19.0000,1.0000,74,74,0.0000,NaN,1,0,0,0,0.0000,0.0000,1.0000


In [25]:
import numpy as np
# ============================================================
# SECTION 2: USER-LEVEL FEATURES (OPTIMIZED)
# ============================================================

# ----------------------------
# Step 1: Score calculation (vectorized)
# ----------------------------
score = (
    np.minimum(user_features['total_events'], 50) / 10 +
    np.minimum(user_features['total_sessions'], 10) / 2 +
    user_features['total_purchases'] * 2
)

# ----------------------------
# Step 2: Segment assignment (FAST)
# ----------------------------
user_features['user_segment'] = np.select(
    [
        score >= 13,
        score >= 10,
        score >= 7,
        score >= 4
    ],
    [
        'power_user',
        'loyal_browser',
        'occasional',
        'fading'
    ],
    default='inactive'
)

# ----------------------------
# Verify
# ----------------------------
print(user_features['user_segment'].value_counts())

user_segment
inactive         1393321
fading              9971
occasional          2643
loyal_browser       1032
power_user           613
Name: count, dtype: int64


In [26]:
import numpy as np

# ============================================================
# SECTION 3: ITEM-LEVEL FEATURES (OPTIMIZED)
# ============================================================

# ----------------------------
# Step 1: Binary flags (FAST)
# ----------------------------
df['is_view'] = (df['event_type'] == 'view').astype(int)
df['is_cart'] = (df['event_type'] == 'addtocart').astype(int)
df['is_purchase'] = (df['event_type'] == 'transaction').astype(int)

# ----------------------------
# Step 2: Item aggregation
# ----------------------------
item_features = df.groupby('itemid').agg(

    item_view_count=('is_view', 'sum'),
    item_cart_count=('is_cart', 'sum'),
    item_purchase_count=('is_purchase', 'sum'),

    item_unique_viewers=('visitorid', 'nunique'),
    item_availability=('available', 'last')

).reset_index()

# ----------------------------
# Step 3: Conversion rates
# ----------------------------
item_features['item_view_to_cart_rate'] = (
    item_features['item_cart_count'] /
    item_features['item_view_count'].replace(0, np.nan)
).round(4)

item_features['item_cart_to_purchase_rate'] = (
    item_features['item_purchase_count'] /
    item_features['item_cart_count'].replace(0, np.nan)
).round(4)

item_features['item_overall_cvr'] = (
    item_features['item_purchase_count'] /
    item_features['item_view_count'].replace(0, np.nan)
).round(4)

# ----------------------------
# Step 4: Ranking
# ----------------------------
item_features['item_popularity_rank'] = (
    item_features['item_view_count']
    .rank(ascending=False, method='dense')
    .astype(int)
)

item_features['item_cvr_rank'] = (
    item_features['item_overall_cvr']
    .fillna(0)
    .rank(ascending=False, method='dense')
    .astype(int)
)

# ----------------------------
# Step 5: 4-Quadrant classification (FAST)
# ----------------------------
median_views = item_features['item_view_count'].median()
median_cvr = item_features['item_overall_cvr'].fillna(0).median()

high_views = item_features['item_view_count'] >= median_views
high_cvr = item_features['item_overall_cvr'].fillna(0) >= median_cvr

item_features['item_quadrant'] = np.select(
    [
        high_views & high_cvr,
        high_views & ~high_cvr,
        ~high_views & high_cvr
    ],
    [
        'star',
        'traffic_waster',
        'hidden_gem'
    ],
    default='dead'
)

# ----------------------------
# Verify
# ----------------------------
print(item_features.shape)
print(item_features['item_quadrant'].value_counts())
item_features.head()

(235061, 12)
item_quadrant
star          126058
hidden_gem    109003
Name: count, dtype: int64


,itemid,item_view_count,item_cart_count,item_purchase_count,item_unique_viewers,item_availability,item_view_to_cart_rate,item_cart_to_purchase_rate,item_overall_cvr,item_popularity_rank,item_cvr_rank,item_quadrant
0,1000,5,0,0,3,-1,0.0000,NaN,0.0000,574,636,star
1,100002,17,1,0,17,0,0.0588,0.0000,0.0000,562,636,star
2,100004,21,2,0,23,1,0.0952,0.0000,0.0000,558,636,star
3,10001,15,0,0,15,-1,0.0000,NaN,0.0000,564,636,star
4,100013,1,0,0,1,-1,0.0000,NaN,0.0000,578,636,hidden_gem


In [27]:
import numpy as np

# ============================================================
# SECTION 4: TIME-BASED FEATURES (OPTIMIZED)
# ============================================================

# ----------------------------
# Step 1: Time flags (FAST vectorized)
# ----------------------------
df['is_evening'] = df['hour_of_day'].between(18, 23).astype(int)
df['is_morning'] = df['hour_of_day'].between(6, 12).astype(int)

# ----------------------------
# Step 2: Days from start
# ----------------------------
dataset_start = df['event_datetime'].min()
df['days_from_start'] = (df['event_datetime'] - dataset_start).dt.days

# ----------------------------
# Step 3: Peak hours (FAST)
# ----------------------------
peak_hours = (
    df.loc[df['event_type'].eq('transaction'), 'hour_of_day']
    .value_counts()
    .nlargest(3)
    .index
    .tolist()
)

# ----------------------------
# Step 4: Peak hour flag
# ----------------------------
df['is_peak_hour'] = df['hour_of_day'].isin(peak_hours).astype(int)

# ----------------------------
# Verify
# ----------------------------
print("Peak purchase hours:", sorted(peak_hours))
print(df[['is_evening', 'is_morning', 'is_peak_hour']].sum())

Peak purchase hours: [0, 1, 23]
is_evening      634816
is_morning      925312
is_peak_hour    544101
dtype: int64


In [28]:
import numpy as np

# ============================================================
# SECTION 5: BEHAVIORAL SIGNAL FEATURES (OPTIMIZED)
# ============================================================

# ----------------------------
# Step 1: Binary flags (FAST)
# ----------------------------
df['is_cart'] = (df['event_type'] == 'addtocart').astype(int)
df['is_purchase'] = (df['event_type'] == 'transaction').astype(int)

# ----------------------------
# Step 2: User aggregation
# ----------------------------
user_behavior = df.groupby('visitorid').agg(

    has_cart_ever=('is_cart', 'max'),
    has_purchase_ever=('is_purchase', 'max'),

    unique_categories=('categoryid', 'nunique'),
    total_sessions_b=('session_id', 'nunique')

).reset_index()

# ----------------------------
# Step 3: Behavioral signals (vectorized)
# ----------------------------

# Cart without purchase
user_behavior['cart_without_purchase'] = (
    (user_behavior['has_cart_ever'] == 1) &
    (user_behavior['has_purchase_ever'] == 0)
).astype(int)

# Multi category browser
user_behavior['multi_category_browser'] = (
    user_behavior['unique_categories'] >= 3
).astype(int)

# Return visitor
user_behavior['return_visitor'] = (
    user_behavior['total_sessions_b'] > 1
).astype(int)

# ----------------------------
# Verify
# ----------------------------
print("Cart without purchase:")
print(user_behavior['cart_without_purchase'].value_counts())

print("\nMulti category browser:")
print(user_behavior['multi_category_browser'].value_counts())

print("\nReturn visitor:")
print(user_behavior['return_visitor'].value_counts())

Cart without purchase:
cart_without_purchase
0    1380434
1      27146
Name: count, dtype: int64

Multi category browser:
multi_category_browser
0    1373597
1      33983
Name: count, dtype: int64

Return visitor:
return_visitor
0    1225947
1     181633
Name: count, dtype: int64


In [29]:
import numpy as np

# ============================================================
# SECTION 5 CONTINUED: TIME TO CART & PURCHASE (FAST)
# ============================================================

# ----------------------------
# Step 1: Split once (FAST, no per-group filtering)
# ----------------------------
views = df[df['event_type'] == 'view'][['visitorid', 'event_datetime']]
carts = df[df['event_type'] == 'addtocart'][['visitorid', 'event_datetime']]
purchases = df[df['event_type'] == 'transaction'][['visitorid', 'event_datetime']]

# ----------------------------
# Step 2: First event times (FAST groupby min)
# ----------------------------
first_events = df.groupby('visitorid', as_index=False).agg(
    first_view_time=('event_datetime', 'min')
)

first_cart = carts.groupby('visitorid', as_index=False).agg(
    first_cart_time=('event_datetime', 'min')
)

first_purchase = purchases.groupby('visitorid', as_index=False).agg(
    first_purchase_time=('event_datetime', 'min')
)

# ----------------------------
# Step 3: Merge (FAST joins)
# ----------------------------
first_events = first_events.merge(first_cart, on='visitorid', how='left')
first_events = first_events.merge(first_purchase, on='visitorid', how='left')

# ----------------------------
# Step 4: Time calculations
# ----------------------------
first_events['time_to_first_cart_mins'] = (
    (first_events['first_cart_time'] - first_events['first_view_time'])
    .dt.total_seconds() / 60
).round(2)

first_events['time_to_first_purchase_mins'] = (
    (first_events['first_purchase_time'] - first_events['first_view_time'])
    .dt.total_seconds() / 60
).round(2)

# ----------------------------
# Verify
# ----------------------------
print(first_events.shape)
first_events.head()

(1407580, 6)


,visitorid,first_view_time,first_cart_time,first_purchase_time,time_to_first_cart_mins,time_to_first_purchase_mins
0,0,2015-09-12 02:49:49.439,NaT,NaT,NaN,NaN
1,1,2015-08-13 23:46:06.444,NaT,NaT,NaN,NaN
2,10,2015-08-05 00:30:29.611,NaT,NaT,NaN,NaN
3,100,2015-09-09 05:48:55.990,NaT,NaT,NaN,NaN
4,1000,2015-07-05 19:38:36.161,NaT,NaT,NaN,NaN


In [30]:
# ============================================================
# SECTION 6: BUILD MASTER FEATURES TABLE (FAST)
# ============================================================

# ----------------------------
# Step 1: Base user table
# ----------------------------
features_master = user_features.copy()

# ----------------------------
# Step 2: Merge behavioral signals (FAST)
# ----------------------------
features_master = features_master.merge(
    user_behavior[['visitorid',
                   'cart_without_purchase',
                   'multi_category_browser',
                   'return_visitor']],
    on='visitorid',
    how='left'
)

# ----------------------------
# Step 3: Merge time-to-event features
# ----------------------------
features_master = features_master.merge(
    first_events[['visitorid',
                  'time_to_first_cart_mins',
                  'time_to_first_purchase_mins']],
    on='visitorid',
    how='left'
)

# ----------------------------
# Step 4: Fill missing values (FAST + clean)
# ----------------------------
fill_cols = [
    'cart_without_purchase',
    'multi_category_browser',
    'return_visitor'
]

features_master[fill_cols] = features_master[fill_cols].fillna(0).astype(int)

# ----------------------------
# Step 5: Verify
# ----------------------------
print(features_master.shape)
features_master.head()

(1407580, 29)


,visitorid,total_sessions,total_events,total_views,total_carts,total_purchases,unique_items_viewed,unique_categories_viewed,first_visit,last_visit,preferred_hour,weekend_session_pct,days_since_first_visit,days_since_last_visit,view_to_cart_ratio,cart_to_purchase_ratio,is_weekend_user,is_repeat_buyer,has_purchased,has_cart,purchase_probability_score,avg_session_duration,avg_events_per_session,user_segment,cart_without_purchase,multi_category_browser,return_visitor,time_to_first_cart_mins,time_to_first_purchase_mins
0,0,1,3,3,0,0,3,3,2015-09-12 02:49:49.439,2015-09-12 02:55:17.175,2.0000,1.0000,6,6,0.0000,NaN,1,0,0,0,0.0000,5.4623,3.0000,inactive,0,1,0,NaN,NaN
1,1,1,1,1,0,0,1,1,2015-08-13 23:46:06.444,2015-08-13 23:46:06.444,23.0000,0.0000,35,35,0.0000,NaN,0,0,0,0,0.0000,0.0000,1.0000,inactive,0,0,0,NaN,NaN
2,10,1,1,1,0,0,1,1,2015-08-05 00:30:29.611,2015-08-05 00:30:29.611,0.0000,0.0000,44,44,0.0000,NaN,0,0,0,0,0.0000,0.0000,1.0000,inactive,0,0,0,NaN,NaN
3,100,1,4,4,0,0,1,1,2015-09-09 05:48:55.990,2015-09-09 05:52:54.884,5.0000,0.0000,9,9,0.0000,NaN,0,0,0,0,0.0000,3.9816,4.0000,inactive,0,0,0,NaN,NaN
4,1000,1,1,1,0,0,1,1,2015-07-05 19:38:36.161,2015-07-05 19:38:36.161,19.0000,1.0000,74,74,0.0000,NaN,1,0,0,0,0.0000,0.0000,1.0000,inactive,0,0,0,NaN,NaN


In [31]:
# ============================================================
# SECTION 7: SUMMARY (OPTIMIZED)
# ============================================================

# ----------------------------
# Basic stats
# ----------------------------
print(f"Total Users     : {len(features_master):,}")
print(f"Total Features  : {features_master.shape[1]}")

# ----------------------------
# Conversion stats
# ----------------------------
print("\n--- Conversion Stats ---")
print(f"Has Purchased   : {features_master['has_purchased'].sum():,}")
print(f"Overall CVR     : {features_master['has_purchased'].mean()*100:.2f}%")
print(f"Repeat Buyers   : {features_master['is_repeat_buyer'].sum():,}")
print(f"Cart Abandoners : {features_master['cart_without_purchase'].sum():,}")

# ----------------------------
# Engagement stats
# ----------------------------
print("\n--- Engagement Stats ---")
print(f"Avg Events/User : {features_master['total_events'].mean():.2f}")
print(f"Avg Sessions    : {features_master['total_sessions'].mean():.2f}")
print(f"Return Visitors : {features_master['return_visitor'].sum():,}")
print(f"Multi-Cat Users : {features_master['multi_category_browser'].sum():,}")

# ----------------------------
# Segment distribution
# ----------------------------
print("\n--- Segment Distribution ---")
print(features_master['user_segment'].value_counts(dropna=False))

# ----------------------------
# Null check (clean version)
# ----------------------------
print("\n--- NULL Check ---")
null_counts = features_master.isnull().sum()
print(null_counts[null_counts > 0])

Total Users     : 1,407,580
Total Features  : 29

--- Conversion Stats ---
Has Purchased   : 11,719
Overall CVR     : 0.83%
Repeat Buyers   : 2,576
Cart Abandoners : 27,146

--- Engagement Stats ---
Avg Events/User : 1.96
Avg Sessions    : 1.25
Return Visitors : 181,633
Multi-Cat Users : 33,983

--- Segment Distribution ---
user_segment
inactive         1393321
fading              9971
occasional          2643
loyal_browser       1032
power_user           613
Name: count, dtype: int64

--- NULL Check ---
view_to_cart_ratio                3401
cart_to_purchase_ratio         1369858
time_to_first_cart_mins        1369858
time_to_first_purchase_mins    1395861
dtype: int64


In [32]:
# ============================================================
# SECTION 8: EXPORT
# ============================================================

# Export master features
features_master.to_csv("features_master.csv", index=False)

# Export session features
session_features.to_csv("session_features.csv", index=False)

# Export item features
item_features.to_csv("item_features.csv", index=False)

# Verify
print("features_master.csv  →", len(features_master), "users")
print("session_features.csv →", len(session_features), "sessions")
print("item_features.csv    →", len(item_features), "items")

features_master.csv  → 1407580 users
session_features.csv → 1761675 sessions
item_features.csv    → 235061 items
